# Create flag parameter


In [0]:
dbutils.widgets.text('incremental_flag', '0')

In [0]:
incremental_flag = dbutils.widgets.get('incremental_flag')


In [0]:
%sql
SELECT * FROM parquet.`abfss://silver-adf@rgstorage.dfs.core.windows.net/car_sales/`

# Create dimension table

In [0]:
df_Dim_Dealer = spark.sql('''
SELECT DISTINCT(Dealer_ID) AS Dealer_ID, DealerName FROM parquet.`abfss://silver-adf@rgstorage.dfs.core.windows.net/car_sales/`
''')


Dim_model sink for initial and incremental load

In [0]:
if spark.catalog.tableExists('sales_catalogue.gold.dim_Dealer'):
   
    df_sink = spark.sql('''
                SELECT Dim_Dealer_key, Dealer_ID, DealerName
                FROM sales_catalogue.gold.dim_Dealer
                ''')
else: 
    
    df_sink = spark.sql('''
                SELECT 1 as Dim_Dealer_key, Dealer_ID, DealerName
                FROM parquet.`abfss://silver-adf@rgstorage.dfs.core.windows.net/car_sales/`
                WHERE 1=0
                ''')

### filtering new and old records

In [0]:
df_filter = df_Dim_Dealer.join(df_sink, df_Dim_Dealer['Dealer_ID'] == df_sink['Dealer_ID'], 'left').select(df_sink['Dim_Dealer_key'],df_Dim_Dealer['Dealer_ID'], df_Dim_Dealer['DealerName'])

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

Df_filter_old

In [0]:
df_filter_old = df_filter.filter(col('Dim_Dealer_key').isNotNull())


df_filter_new

In [0]:
df_filter_new = df_filter.filter(col('Dim_Dealer_key').isNull())


# Create surrogate key

fetch the max surrogate key from the column from the existing table

In [0]:
if (incremental_flag == '0'):
    max_value = 1
else:
    max_value_df = spark.sql("select max(Dim_Dealer_key) from sales_catalogue.gold.dim_Dealer")
    max_value = max_value_df.collect()[0][0]

# Create surrogate key column and add the max surrogate key

In [0]:
df_filter_new = df_filter_new.withColumn('Dim_Dealer_key', max_value+monotonically_increasing_id())

In [0]:
df_filter_new.display()


Dim_Dealer_key,Dealer_ID,DealerName


FINAL JOIN NEW + OLD

In [0]:
df_final = df_filter_old.union(df_filter_new)

In [0]:
df_final.display()

Dim_Dealer_key,Dealer_ID,DealerName
1,DLR0058,Fiat do Brasil Motors
2,DLR0107,Land Rover Motors
3,DLR0129,Mia Motors
4,DLR0111,Lotus Motors
5,DLR0085,Humber Motors
6,DLR0001,AC Cars Motors
7,DLR0218,Lagonda Motors
8,DLR0082,Honda Motors
9,DLR0063,Ford do Brasil Motors
10,DLR0193,Tazzari Motors


SCD- 1 Upsert

In [0]:
from delta.tables import *

In [0]:
if spark.catalog.tableExists('sales_catalog.gold.dim_Dealer'):
    delta_table = deltaTable.forPath("abfss://gold-adf@rgstorage.dfs.core.windows.net/car_sales_dim_Dealer")
    delta_table.alias("trg").merge(df_final.alias("src"), "trg.Dim_Dealer_key = src.Dim_Dealer_key")\
                             .whenMatchedUpdateAll()\
                             .whenNotMatchedInsertAll()\
                             .execute()

else: 
    df_final.write.format('delta')\
        .mode('overwrite')\
            .option("path", "abfss://gold-adf@rgstorage.dfs.core.windows.net/car_sales_dim_Dealer")\
            .saveAsTable('sales_catalogue.gold.dim_Dealer')

In [0]:
%sql
select * from sales_catalogue.gold.dim_Dealer